# 01 Data and Target

**Reproducibility note:** This notebook loads committed summary artifacts generated by the full modeling workflow. It performs lightweight calculations directly from those artifacts and provides an interpretive walkthrough of the target definition and resolved-loan sample.

### Objective
Define the credit outcome used for PD modeling and summarize the resolved LendingClub loan sample.

### Inputs
- `reports/tables/default_rate_by_grade.csv`
- `reports/tables/default_rate_by_term.csv`
- `data/processed/expected_loss_by_grade_v1.csv`

### Methodology
Resolved loans are mapped to a binary target where `Fully Paid = 0` and `Charged Off / Default = 1`. Unresolved statuses are excluded from the supervised modeling sample.

### Key Outputs
- Target definition mapping
- Default-rate summaries by grade and term


In [1]:
from pathlib import Path
import sys
import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root / 'src'))

reports = project_root / 'reports'
tables = reports / 'tables'
processed = project_root / 'data' / 'processed'

from credit_risk.data import DEFAULT_STATUS_MAP

DEFAULT_STATUS_MAP


{'Fully Paid': 0, 'Charged Off': 1, 'Default': 1}

In [2]:
grade = pd.read_csv(tables / 'default_rate_by_grade.csv')
term = pd.read_csv(tables / 'default_rate_by_term.csv')
portfolio = pd.read_csv(processed / 'expected_loss_by_grade_v1.csv')

summary = pd.DataFrame({
    'metric': ['loans', 'defaults', 'observed_default_rate'],
    'value': [portfolio['loan_count'].sum(), portfolio['default_count'].sum(), portfolio['default_count'].sum() / portfolio['loan_count'].sum()],
})
summary


,metric,value
0,loans,1.345350e+06
1,defaults,2.685990e+05
2,observed_default_rate,1.996499e-01


In [3]:
grade.sort_values('grade')


,grade,number_of_loans,number_of_defaults,default_rate
6,A,235095,14206,0.060427
5,B,392748,52576,0.133867
4,C,381694,85657,0.224413
3,D,200966,61067,0.303867
2,E,93656,36041,0.384823
1,F,32059,14492,0.452042
0,G,9132,4560,0.499343


In [4]:
term


,term,number_of_loans,number_of_defaults,default_rate
0,60,324582,105322,0.324485
1,36,1020768,163277,0.159955


### Findings
- The modeling sample contains resolved loan outcomes only.
- Default rates increase materially across LendingClub grades.
- Longer-term loans show higher realized default rates than shorter-term loans.

### Limitations
- The sample includes accepted loans only and does not represent rejected applications.
- Excluding unresolved loans creates a resolved-outcome modeling sample rather than a censoring-aware default-time dataset.
